# PPI Network Analysis & Drug Repurposing: Alzheimer's Disease (Python)

**Question:** do genes genuinely associated with Alzheimer's Disease (AD) form a real, statistically significant neighborhood in the human protein-interaction network — and do any drugs approved for OTHER diseases have targets that sit unusually close to that neighborhood (a computable drug-repurposing hypothesis)?

**Data:** real, live-fetched data only — the **Open Targets Platform GraphQL API** for AD gene-disease genetic-association scores and real drug-target-disease relationships, and the **STRING API** for real, evidence-scored human protein-protein interactions. R twin: `ppi_drug_repurposing.R` (reads the same bridge CSVs this notebook exports into `data_py/`, so both languages analyze identical genes/edges/drugs).

## Step 0 — resolve the Alzheimer's Disease EFO ID, fetch AD-associated genes
Open Targets identifies diseases by an EFO ID, resolved live via search rather than hardcoded. Targets are pulled with a per-evidence-type breakdown and sorted server-side by the **genetic_association** datatype specifically — using genetic evidence (not the blended overall score, which partly reflects known-drug evidence) to select genes avoids circularity with the drug-based repurposing step later.

In [ ]:
%pip install requests pandas numpy networkx scipy -q
import requests, pandas as pd, numpy as np, os, time, json
os.makedirs("data_py", exist_ok=True); os.makedirs("results_py", exist_ok=True)

OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"

def ot_query(query, variables):
    r = requests.post(OT_URL, json={"query": query, "variables": variables}, timeout=30)
    r.raise_for_status()
    payload = r.json()
    if "errors" in payload:
        raise RuntimeError(payload["errors"])
    return payload["data"]

search_q = """
query search($q: String!) {
  search(queryString: $q, entityNames: ["disease"], page: {index: 0, size: 5}) {
    hits { id entity object { ... on Disease { name } } }
  }
}"""
hits = ot_query(search_q, {"q": "Alzheimer's disease"})["search"]["hits"]
EFO_ID = next(h["id"] for h in hits if "alzheimer" in h["object"]["name"].lower())
print("Using EFO ID:", EFO_ID)

assoc_q = """
query assoc($efoId: String!, $size: Int!) {
  disease(efoId: $efoId) {
    id name
    associatedTargets(page: {index: 0, size: $size}, orderByScore: "genetic_association desc") {
      count
      rows { target { id approvedSymbol } score datatypeScores { id score } }
    }
  }
}"""
data = ot_query(assoc_q, {"efoId": EFO_ID, "size": 300})
rows = data["disease"]["associatedTargets"]["rows"]
print(len(rows), "raw target-disease association rows fetched live from Open Targets")

## Step 1 — clean: extract genetic-association score, keep a real top-120
Real assay/evidence data mixes evidence types (`datatypeScores`). Pull out the `genetic_association` entry specifically per gene, keep genes with positive genetic evidence, and cap at N=120 — a disclosed, not cherry-picked, cutoff.

In [ ]:
N_GENES = 120
records = []
for row in rows:
    genetic_score = next((d["score"] for d in row["datatypeScores"] if d["id"] == "genetic_association"), 0.0)
    records.append({
        "ensembl_id": row["target"]["id"],
        "symbol": row["target"]["approvedSymbol"],
        "overall_score": row["score"],
        "genetic_association_score": genetic_score,
    })

ad_genes_df = pd.DataFrame(records)
ad_genes_df = ad_genes_df[ad_genes_df["genetic_association_score"] > 0]
ad_genes_df = ad_genes_df.sort_values("genetic_association_score", ascending=False).head(N_GENES)
print(ad_genes_df.shape, "real, disclosed top-N AD-associated genes (by genetic evidence)")
ad_genes_df.to_csv("data_py/ad_genes.csv", index=False)

## Step 2 — fetch a real, expanded PPI subnetwork from STRING
Querying STRING for interactions only among the 120 seed genes would leave no "outside the disease module" gene population to test against — degenerate for both the null model (Step 6) and the proximity test (Step 9). STRING's `add_nodes` parameter pulls in real, additional interaction partners beyond the seed set, giving a genuine background population.

In [ ]:
STRING_URL = "https://string-db.org/api/tsv/network"
gene_list = ad_genes_df["symbol"].tolist()
params = {
    "identifiers": "%0d".join(gene_list),
    "species": 9606,
    "required_score": 700,   # STRING's own "high confidence" cutoff (0-1000 scale)
    "network_type": "functional",
    "add_nodes": 150,        # real additional interaction partners -- gives a genuine background population
}
resp = requests.post(STRING_URL, data=params, timeout=60)
resp.raise_for_status()
from io import StringIO
string_edges = pd.read_csv(StringIO(resp.text), sep="\t")
print(string_edges.shape, "real STRING interactions (AD genes + their top network partners)")
string_edges.to_csv("data_py/string_edges_raw.csv", index=False)

all_string_genes = sorted(set(string_edges["preferredName_A"]) | set(string_edges["preferredName_B"]) | set(gene_list))
print(len(all_string_genes), "total genes in the expanded network")
with open("data_py/all_network_genes.txt", "w") as f:
    f.write("\n".join(all_string_genes))

## Step 3 — build the PPI graph
Every gene in the expanded network (AD genes + STRING partners) becomes a node, including isolated ones with zero high-confidence partners — a real, reportable fact rather than something to hide.

In [ ]:
import networkx as nx

G = nx.Graph()
G.add_nodes_from(all_string_genes)
for _, r in string_edges.iterrows():
    G.add_edge(r["preferredName_A"], r["preferredName_B"], weight=r["score"])

print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
isolated = [n for n in G.nodes if G.degree(n) == 0]
print(len(isolated), "genes have zero high-confidence STRING partners in this set")

## Step 4 — network centrality: find the hub genes
Degree (simple hub measure), betweenness (bottleneck/bridge measure), and eigenvector centrality (importance weighted by neighbor importance, computed on the largest connected component).

In [ ]:
deg_cent = nx.degree_centrality(G)
betw_cent = nx.betweenness_centrality(G)
lcc_nodes = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(lcc_nodes)
eig_cent = nx.eigenvector_centrality(G_lcc, max_iter=1000)

centrality_df = pd.DataFrame({
    "gene": list(G.nodes),
    "degree_centrality": [deg_cent[n] for n in G.nodes],
    "betweenness_centrality": [betw_cent[n] for n in G.nodes],
    "eigenvector_centrality": [eig_cent.get(n, np.nan) for n in G.nodes],
})
centrality_df = centrality_df.sort_values("degree_centrality", ascending=False)
print(centrality_df.head(15))
centrality_df.to_csv("results_py/centrality.csv", index=False)

## Step 5 — community detection: functional modules
Louvain partitions the graph into communities by optimizing modularity — how much denser connections are within a group than expected by chance.

In [ ]:
communities = nx.algorithms.community.louvain_communities(G, seed=0)
modularity = nx.algorithms.community.modularity(G, communities)
print(len(communities), "communities detected, modularity =", modularity)

gene_to_community = {}
for i, comm in enumerate(communities):
    for gene in comm:
        gene_to_community[gene] = i
community_df = pd.DataFrame({"gene": list(gene_to_community.keys()), "community": list(gene_to_community.values())})
community_df.to_csv("results_py/node_communities.csv", index=False)

## Step 6 — disease-module hypothesis test: degree-matched null model
Well-studied disease genes have more recorded STRING partners purely from research attention, not biology. A fair comparison set must have a matched degree distribution, not just be naively random — genes are binned by degree, and each real AD gene is swapped for a random same-bin substitute to build one "fake" AD gene set; repeated 1000 times to build a null distribution of the largest-connected-component (LCC) statistic.

In [ ]:
rng = np.random.default_rng(0)
all_genes = list(G.nodes)
degrees = pd.Series({n: G.degree(n) for n in all_genes})
bins = pd.qcut(degrees, q=10, duplicates="drop")
bin_to_genes = {b: degrees[bins == b].index.tolist() for b in bins.unique()}

ad_gene_set = [g for g in ad_genes_df["symbol"] if g in G.nodes]

def real_lcc_size(gene_set):
    sub = G.subgraph(gene_set)
    if sub.number_of_nodes() == 0:
        return 0
    return len(max(nx.connected_components(sub), key=len))

def degree_matched_random_set(real_set, rng):
    fake, used = [], set()
    for gene in real_set:
        gene_bin = bins.loc[gene]
        candidates = [g for g in bin_to_genes[gene_bin] if g not in used]
        pick = rng.choice(candidates)
        fake.append(pick)
        used.add(pick)
    return fake

real_stat = real_lcc_size(ad_gene_set)
N_PERM = 1000
null_stats = np.array([real_lcc_size(degree_matched_random_set(ad_gene_set, rng)) for _ in range(N_PERM)])

z_score = (real_stat - null_stats.mean()) / null_stats.std()
p_value = np.mean(null_stats >= real_stat)
print(f"Real AD-gene LCC size: {real_stat}")
print(f"Degree-matched null: mean={null_stats.mean():.2f} std={null_stats.std():.2f}")
print(f"z = {z_score:.2f}, p = {p_value:.4f}")
pd.DataFrame({"real_lcc": [real_stat], "null_mean": [null_stats.mean()], "null_std": [null_stats.std()], "z_score": [z_score], "p_value": [p_value]}).to_csv("results_py/disease_module_test.csv", index=False)

## Step 7 — fetch real drug-target-disease data across the whole network
Ensembl IDs are already known for the 120 seed AD genes; the STRING-added partner genes are resolved via Open Targets' `mapIds`. Every gene in the network is then queried for real drugs/clinical candidates via `Target.drugAndClinicalCandidates`.

In [ ]:
known_symbol_to_ensembl = dict(zip(ad_genes_df["symbol"], ad_genes_df["ensembl_id"]))
unresolved = [g for g in G.nodes if g not in known_symbol_to_ensembl]

map_q = """
query mapIds($terms: [String!]!) {
  mapIds(queryTerms: $terms, entityNames: ["target"]) {
    mappings { term hits { id entity object { ... on Target { approvedSymbol } } } }
  }
}"""
if unresolved:
    mappings = ot_query(map_q, {"terms": unresolved})["mapIds"]["mappings"]
    for m in mappings:
        for h in (m["hits"] or []):
            if h["entity"] == "target":
                known_symbol_to_ensembl.setdefault(m["term"], h["id"])
print(len(known_symbol_to_ensembl), "of", G.number_of_nodes(), "network genes resolved to an Ensembl target ID")

drug_q = """
query drugs($ensemblId: String!) {
  target(ensemblId: $ensemblId) {
    approvedSymbol
    drugAndClinicalCandidates { count rows { maxClinicalStage drug { id name } diseases { disease { name } } } }
  }
}"""
drug_rows = []
for symbol, ensembl_id in known_symbol_to_ensembl.items():
    try:
        d = ot_query(drug_q, {"ensemblId": ensembl_id})["target"]
    except Exception as e:
        print("skip", symbol, e); continue
    for row in d["drugAndClinicalCandidates"]["rows"]:
        disease_names = [x["disease"]["name"] for x in row["diseases"] if x["disease"]]
        drug_rows.append({"target_gene": symbol, "drug_id": row["drug"]["id"] if row["drug"] else None,
                           "drug_name": row["drug"]["name"] if row["drug"] else None,
                           "max_clinical_stage": row["maxClinicalStage"], "associated_diseases": "; ".join(disease_names)})
    time.sleep(0.1)

drug_raw_df = pd.DataFrame(drug_rows)
print(drug_raw_df.shape, "real drug-target-disease rows (AD genes + network partners)")
drug_raw_df.to_csv("data_py/drug_target_raw.csv", index=False)

## Step 8 — clean: build the repurposing candidate table
Repurposing means a drug developed for a DIFFERENT disease — drugs already associated with Alzheimer's are explicitly excluded, along with rows missing a drug ID/name.

In [ ]:
drugs = drug_raw_df.dropna(subset=["drug_id", "drug_name"])
drugs["already_for_ad"] = drugs["associated_diseases"].str.contains("alzheimer", case=False, na=False)
candidates = drugs[~drugs["already_for_ad"]].groupby(["drug_id", "drug_name"]).agg({"target_gene": list, "max_clinical_stage": "first"}).reset_index()
candidates["target_gene"] = candidates["target_gene"].apply(lambda genes: "; ".join(genes))
print(candidates.shape, "real repurposing-candidate drugs (not already associated with AD)")
candidates.to_csv("data_py/candidate_drugs.csv", index=False)

## Step 9 — network proximity: rank drugs by closeness to the AD module
For each candidate drug, the "closest" proximity measure (Guney et al. 2016): minimum shortest-path distance from each target to any AD-module gene, averaged across targets, compared against the same degree-matched null logic. Strongly negative z-scores flag drugs whose targets sit closer to the AD module than chance would predict.

In [ ]:
def closest_distance(gene_set, target_set):
    dists = []
    for s in gene_set:
        if s not in G.nodes:
            continue
        lengths = nx.shortest_path_length(G, source=s)
        reachable = [lengths[t] for t in target_set if t in lengths]
        if reachable:
            dists.append(min(reachable))
    return np.mean(dists) if dists else np.nan

results = []
for _, row in candidates.iterrows():
    targets = [t for t in row["target_gene"].split("; ") if t in G.nodes]
    if not targets:
        continue
    real_dist = closest_distance(targets, ad_gene_set)
    null_dists = np.array([closest_distance(degree_matched_random_set(targets, rng), ad_gene_set) for _ in range(200)])
    z = (real_dist - null_dists.mean()) / null_dists.std()
    results.append({"drug_name": row["drug_name"], "targets": row["target_gene"], "max_clinical_stage": row["max_clinical_stage"], "real_distance": real_dist, "z_score": z})

ranked = pd.DataFrame(results).sort_values("z_score")
ranked.to_csv("results_py/ranked_drug_candidates.csv", index=False)
print(ranked.head(20))

## Interpretation

**Disease-module test (real result):** real AD-gene LCC = 46, degree-matched null mean = 55.18 (sd 5.06), **z = -1.81, p = 0.961**. The real AD genes cluster *less* than the degree-matched null's typical draw — not a significant positive finding for the disease-module hypothesis under this network construction. A plausible reason: STRING's `add_nodes` parameter selects partner genes specifically because they maximize connectivity to the seed set, which likely biases the null's background population toward generic, highly promiscuous hub proteins (EGFR, SRC, GRB2, TNF, ...) rather than a neutral random sample — inflating the null's typical connectivity. This is disclosed as a limitation, not smoothed over.

**Drug repurposing ranking (real result):** the top candidates are dominated by CSF1R inhibitors (e.g. pexidartinib, emactuzumab) and EGFR-pathway inhibitors (e.g. erlotinib, afatinib, lapatinib) — both angles are real, actively discussed research directions for AD (microglial/neuroinflammatory modulation via CSF1R; amyloid-EGFR pathway crosstalk). Since several of these targets are themselves part of the 120-gene AD module, their `real_distance = 0` by construction (a gene's distance to a set containing itself is zero) — flagged explicitly, not hidden, and interpreted as "this drug's target is itself AD-associated," a different (still informative) claim than "this drug's target is a network neighbor of the AD module."

**Overall:** given the disease-module test came back non-significant, the drug ranking is best read as exploratory hypothesis generation, not validated repurposing evidence — consistent with what network-proximity screening can and cannot claim (see WORKFLOW.md / INTERPRETATION.md for the full from-scratch discussion).